# 08. Self-Attention, Causal Mask, Multi-Head Attention

07에서는 Query = Decoder 상태, Key = Value = Encoder 출력이었다. 이것을 일반화한다.

- 입력 $X$를 학습 가능한 행렬로 투영해서 $Q = XW_q,\ K = XW_k,\ V = XW_v$ 를 만든다.
- **Self-Attention**: $Q, K, V$가 모두 같은 시퀀스에서 나온다. 각 토큰이 다른 모든 토큰의 정보를 **한 번에(병렬로)** 섞는다.

$$\text{Attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

## 1. Scaled Dot-Product Attention

In [2]:
def attention(Q, K, V, mask=None):
    d_k = Q.size(-1)
    scores = Q @ K.transpose(-2, -1) / d_k ** 0.5        # (..., Tq, Tk)
    if mask is not None:
        scores = scores.masked_fill(mask, float("-inf"))  # mask가 True인 곳을 가림
    weights = torch.softmax(scores, dim=-1)               # 각 Query 행의 합 = 1
    return weights @ V, weights                           # (..., Tq, d_v)


Q, K, V = torch.randn(2, 5, 8), torch.randn(2, 5, 8), torch.randn(2, 5, 8)
out, w = attention(Q, K, V)
print("out:", out.shape, "| weights:", w.shape)
print("행 합:", w[0].sum(dim=-1))
print("PyTorch 내장 F.scaled_dot_product_attention과 일치:",
      torch.allclose(out, F.scaled_dot_product_attention(Q, K, V), atol=1e-6))

out: torch.Size([2, 5, 8]) | weights: torch.Size([2, 5, 5])
행 합: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000])
PyTorch 내장 F.scaled_dot_product_attention과 일치: True


## 2. 왜 $\sqrt{d_k}$로 나누나

평균 0, 분산 1인 벡터 두 개를 내적하면 분산이 $d_k$가 된다. 차원이 크면 점수 차이가 커져서 softmax가 거의 one-hot이 되고,
그러면 **기울기가 0에 가까워져** 학습이 멈춘다. $\sqrt{d_k}$로 나누면 분산이 다시 1이 된다.

In [3]:
for d in [4, 64, 512]:
    q, k = torch.randn(10000, d), torch.randn(10000, d)
    dots = (q * k).sum(dim=-1)
    scores = torch.randn(1000, 10, d) @ torch.randn(1000, d, 1)          # 키 10개에 대한 점수
    raw_max = torch.softmax(scores.squeeze(-1), dim=-1).max(dim=-1).values.mean()
    scaled_max = torch.softmax(scores.squeeze(-1) / d ** 0.5, dim=-1).max(dim=-1).values.mean()
    print(f"d={d:3d}  내적 분산 {dots.var():7.1f}  |  softmax 최댓값 평균: 그대로 {raw_max:.2f}, √d로 나눔 {scaled_max:.2f}")

d=  4  내적 분산     4.0  |  softmax 최댓값 평균: 그대로 0.50, √d로 나눔 0.31
d= 64  내적 분산    63.9  |  softmax 최댓값 평균: 그대로 0.86, √d로 나눔 0.32


d=512  내적 분산   499.8  |  softmax 최댓값 평균: 그대로 0.95, √d로 나눔 0.32


## 3. Self-Attention 모듈 (Q, K, V 투영)

In [4]:
class SelfAttention(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, mask=None):                 # x: (B, T, d_model)
        return attention(self.W_q(x), self.W_k(x), self.W_v(x), mask)


torch.manual_seed(0)
sa = SelfAttention(8)
x = torch.randn(1, 4, 8)
out, w = sa(x)
print("out:", out.shape)
print("weights (토큰 i가 토큰 j를 보는 비율):\n", w[0].detach().round(decimals=2))

out: torch.Size([1, 4, 8])
weights (토큰 i가 토큰 j를 보는 비율):
 tensor([[0.2500, 0.2300, 0.2700, 0.2600],
        [0.3000, 0.3000, 0.2100, 0.2000],
        [0.2600, 0.3000, 0.2400, 0.2000],
        [0.2100, 0.2800, 0.3000, 0.2100]])


## 4. Self-Attention에는 순서 정보가 없다

토큰 순서를 섞으면 출력도 **똑같이 섞일 뿐** 값은 그대로다. 즉 "개가 사람을 문다"와 "사람이 개를 문다"를 구분하지 못한다.
그래서 Transformer는 **위치 임베딩**을 입력에 더한다 (09).

In [5]:
perm = torch.tensor([2, 0, 3, 1])
out_perm, _ = sa(x[:, perm])
print("sa(섞은 x) == 섞은 sa(x):", torch.allclose(out_perm, out[:, perm], atol=1e-6))

sa(섞은 x) == 섞은 sa(x): True


## 5. Causal Mask — 미래를 보지 못하게

GPT처럼 다음 토큰을 예측하는 모델은 학습할 때 정답(미래 토큰)을 보면 안 된다.
대각선 **위쪽**(j > i)의 점수를 `-inf`로 만들면 softmax 결과가 0이 된다.

In [6]:
T = 4
mask = torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)
print(mask.int())

out_c, w_c = sa(x, mask)
print(w_c[0].detach().round(decimals=2))           # 아래 삼각만 값이 있음

q, k, v = sa.W_q(x), sa.W_k(x), sa.W_v(x)
print("is_causal=True 내장 함수와 일치:",
      torch.allclose(out_c, F.scaled_dot_product_attention(q, k, v, is_causal=True), atol=1e-6))

# 마지막 토큰을 바꿔도 앞쪽 출력은 변하지 않는다
x2 = x.clone()
x2[:, -1] += 10
out_c2, _ = sa(x2, mask)
print("앞 3개 출력 그대로:", torch.allclose(out_c[:, :-1], out_c2[:, :-1], atol=1e-6),
      "| 마지막 출력 변함:", not torch.allclose(out_c[:, -1], out_c2[:, -1]))

tensor([[0, 1, 1, 1],
        [0, 0, 1, 1],
        [0, 0, 0, 1],
        [0, 0, 0, 0]], dtype=torch.int32)
tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000],
        [0.3200, 0.3800, 0.3000, 0.0000],
        [0.2100, 0.2800, 0.3000, 0.2100]])
is_causal=True 내장 함수와 일치: True
앞 3개 출력 그대로: True | 마지막 출력 변함: True


## 6. Multi-Head Attention

Attention을 하나만 쓰지 않고, 차원을 `h`개로 나눠 **서로 다른 관점(head)으로 병렬로** 계산한 뒤 이어 붙인다.
예를 들어 어떤 head는 바로 앞 단어를, 어떤 head는 주어를 보는 식으로 역할이 나뉜다.

| 단계 | shape |
| --- | --- |
| 입력 `x` | `(B, T, C)` |
| `W_q(x)` | `(B, T, C)` |
| `.view(B, T, h, d_head)` | `(B, T, h, d_head)` — `C = h × d_head` |
| `.transpose(1, 2)` | `(B, h, T, d_head)` — head를 배치처럼 취급 |
| `q @ k.transpose(-2, -1)` | `(B, h, T, T)` |
| `weights @ v` | `(B, h, T, d_head)` |
| `.transpose(1, 2).contiguous().view(B, T, C)` | `(B, T, C)` — head 이어 붙이기 |
| `out_proj` | `(B, T, C)` |

In [7]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.h, self.d_head = n_heads, d_model // n_heads
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, x, causal=False):
        B, T, C = x.shape
        def split(t):                                            # (B, T, C) → (B, h, T, d_head)
            return t.view(B, T, self.h, self.d_head).transpose(1, 2)
        q, k, v = split(self.W_q(x)), split(self.W_k(x)), split(self.W_v(x))
        mask = torch.triu(torch.ones(T, T, dtype=torch.bool, device=x.device), diagonal=1) if causal else None
        out, w = attention(q, k, v, mask)                        # (B, h, T, d_head)
        out = out.transpose(1, 2).contiguous().view(B, T, C)     # head 이어 붙이기 → (B, T, C)
        return self.out_proj(out), w


torch.manual_seed(0)
x = torch.randn(2, 6, 16)
mha = MultiHeadAttention(d_model=16, n_heads=4)
out, w = mha(x, causal=True)
print("out:", out.shape, "| weights:", w.shape)     # (2, 6, 16), (2, 4, 6, 6)

out: torch.Size([2, 6, 16]) | weights: torch.Size([2, 4, 6, 6])


PyTorch 내장 `nn.MultiheadAttention`에 같은 가중치를 넣어서 결과가 같은지 확인한다.
(내장 구현은 $W_q, W_k, W_v$를 하나로 이어 붙인 `in_proj_weight`로 가지고 있다.)

In [8]:
ref = nn.MultiheadAttention(16, 4, batch_first=True)
with torch.no_grad():
    ref.in_proj_weight.copy_(torch.cat([mha.W_q.weight, mha.W_k.weight, mha.W_v.weight]))
    ref.in_proj_bias.zero_()
    ref.out_proj.weight.copy_(mha.out_proj.weight)
    ref.out_proj.bias.copy_(mha.out_proj.bias)

causal_mask = torch.triu(torch.ones(6, 6, dtype=torch.bool), diagonal=1)
ref_out, _ = ref(x, x, x, attn_mask=causal_mask)
print("nn.MultiheadAttention과 일치:", torch.allclose(out, ref_out, atol=1e-5))

nn.MultiheadAttention과 일치: True
